# C1-M2 — Same-Day PIT Reader: G1 (No Look-Ahead) and G2 (Train/Serve Parity)

End-to-end validation (METHODOLOGY §17) for the C1-M2 deliverable on the **real
lake**. It exercises the cross-module path
`storage/realtime.get_pit_panel` → `features.engineering.build_features(asof)` →
batch-parity check, and renders the two merge-blocking gate verdicts.

**Reference documents**
- PRD: [`c1-live-data.prd.md`](../.claude/prds/c1-live-data.prd.md) — Success Metrics G1/G2.
- Contract: [`docs/concepts/data-freshness-slas.md`](../docs/concepts/data-freshness-slas.md)
  — the frozen processed-only / as-of-instant / weekend read decisions (C1-M1).
- Module: [`src/quant/storage/realtime.py`](../src/quant/storage/realtime.py) —
  the gate functions and pinned thresholds are the source of truth (METHODOLOGY §2).

**Gates (pinned in code; this notebook quotes the function output verbatim)**
- **G1 — no look-ahead.** `get_pit_bar(symbol, asof)` never returns a bar whose
  `timestamp > asof`, across a sweep of intraday/weekend as-of instants —
  future-bar count must be **exactly 0**.
- **G2 — train/serve parity.** Features built from an `asof`-truncated history
  equal the full-history batch features for the retained date, to within
  `rtol <= 1e-9` with **0 material mismatches**, over **>= 250** `(symbol, date)`
  pairs spanning **>= 2 regimes** (COVID 2020 + rate-hike cycle 2022-23).

This is infrastructure, not a research trial — no Sharpe claim, no ledger
deflation contribution (C1 PRD "Ledger discipline").

In [ ]:
import pandas as pd

from quant.features.engineering import build_features
from quant.storage.realtime import (
    PARITY_MIN_PAIRS,
    PARITY_MIN_REGIMES,
    PARITY_RTOL,
    get_pit_bar,
    get_pit_panel,
    parity_gate_report,
    pit_gate_report,
)

# Six symbols with full 2001-2026 history in the lake.
SYMBOLS = ["BA", "JNJ", "MCD", "CSCO", "AMZN", "NVDA"]

# "Now" for assembling the full-history ground-truth panel. Any instant after
# the last stored bar yields the complete history.
NOW = pd.Timestamp("2026-06-27", tz="UTC")

full_panel = get_pit_panel(SYMBOLS, NOW)
batch_feats = build_features(SYMBOLS, full_panel)  # full-history ground truth
print({s: full_panel[s].shape for s in SYMBOLS})
print("feature columns:", list(batch_feats[SYMBOLS[0]].columns))

## G1 - No look-ahead

Sweep as-of instants across the COVID window for three symbols, at three
offsets relative to each calendar day: **-2h** (just before the 00:00 UTC bar
stamp, the off-by-one trap), **+6h** (just after), and **+30h** (next day /
weekend). Every returned bar must be stamped `<= asof`.

In [ ]:
g1_checks = []
sweep_days = pd.date_range("2020-02-01", "2020-05-01", freq="1D", tz="UTC")
offsets = [pd.Timedelta(-2, "h"), pd.Timedelta(6, "h"), pd.Timedelta(30, "h")]
for sym in SYMBOLS[:3]:
    for day in sweep_days:
        for off in offsets:
            asof = day + off
            bar = get_pit_bar(sym, asof)
            g1_checks.append((asof, None if bar is None else bar.name))

g1 = pit_gate_report(g1_checks)
print(g1)

## G2 - Train/serve parity

For each sampled session `T` in two regimes, read the **truncated** history via
`get_pit_panel(SYMBOLS, T)`, rebuild features, and compare each symbol's last
row (the live row for `T`) against the full-history batch row for `T`. This is
the genuine reader path - a fresh as-of query per date, not just the in-process
`asof` truncation - and it exercises the FRED publication-lag parity lever on
real macro data.

In [ ]:
def _sample(idx, start, end, k):
    win = idx[(idx >= start) & (idx <= end)]
    step = max(1, len(win) // k)
    return list(win[::step][:k])

idx = batch_feats[SYMBOLS[0]].index
covid = _sample(idx, pd.Timestamp("2020-02-01", tz="UTC"), pd.Timestamp("2020-06-30", tz="UTC"), 25)
rate = _sample(idx, pd.Timestamp("2022-06-01", tz="UTC"), pd.Timestamp("2023-06-30", tz="UTC"), 25)
sampled = covid + rate
print(f"sampled {len(sampled)} sessions across 2 regimes")

batch_rows, live_rows = {}, {}
for T in sampled:
    live_panel = get_pit_panel(SYMBOLS, T)
    live_feats = build_features(SYMBOLS, live_panel)
    for sym in SYMBOLS:
        if T not in batch_feats[sym].index or live_feats.get(sym) is None or live_feats[sym].empty:
            continue
        key = f"{sym}|{T.isoformat()}"
        batch_rows[key] = batch_feats[sym].loc[T]
        live_rows[key] = live_feats[sym].iloc[-1]

batch_df = pd.DataFrame(batch_rows).T
live_df = pd.DataFrame(live_rows).T
g2 = parity_gate_report(batch_df, live_df, n_regimes=2)
print(g2)

## Verdict

Both gates are deterministic correctness predicates. A PASS means the same-day
data path is point-in-time-correct (G1) **and** bit-for-bit identical to the
backtest path (G2) - a deployment-safe reader. A G2 FAIL would be a valid
pre-committed negative that **blocks C2** until the divergence is reconciled
(C1 PRD).

In [ ]:
verdict = pd.DataFrame(
    [
        {"gate": "G1 no-look-ahead", "metric": "future bars", "value": g1.n_future_bars,
         "threshold": "== 0", "pass": g1.passed},
        {"gate": "G2 parity (pairs)", "metric": "pairs", "value": g2.n_pairs,
         "threshold": f">= {PARITY_MIN_PAIRS}", "pass": g2.n_pairs >= PARITY_MIN_PAIRS},
        {"gate": "G2 parity (regimes)", "metric": "regimes", "value": g2.n_regimes,
         "threshold": f">= {PARITY_MIN_REGIMES}", "pass": g2.n_regimes >= PARITY_MIN_REGIMES},
        {"gate": "G2 parity (mismatch)", "metric": "mismatches", "value": g2.n_mismatches,
         "threshold": "== 0", "pass": g2.n_mismatches == 0},
        {"gate": "G2 parity (rtol)", "metric": "max rel diff", "value": g2.max_rel_diff,
         "threshold": f"<= {PARITY_RTOL}", "pass": g2.max_rel_diff <= PARITY_RTOL},
    ]
)
print(verdict.to_string(index=False))

assert g1.passed, "G1 FAILED - reader leaked a future bar"
assert g2.passed, "G2 FAILED - train/serve skew present; C2 stays blocked"
print("\nC1-M2 GATE: PASS - same-day reader is PIT-correct (G1) and batch-identical (G2).")

## Audit-only ledger entry (C1-M2-LEDGER-AUDIT)

The C1 PRD's *Ledger discipline* says a C1 milestone run that emits a verdict
artifact **may** record an **audit-only** ledger entry (`n_comparisons = 0`) per
the A-LEDGER-RUNNERS pattern — bookkeeping for audit symmetry with the Phase 4A /
B1 / C2-M3 runner integrations, **not** a deflation contribution (C1 makes no
Sharpe claim).

The cell below records that entry via the `quant.ledger` library. It is
**idempotent** (`record_run(skip_if_exists=True)`, keyed on `config_hash`) and has
no network dependency, so it does not require re-running this notebook's live-data
gate cells. The committed entry is `ledger-2026-06-29-0001` in
[`data/ledger.yaml`](../data/ledger.yaml).

In [ ]:
# C1-M2-LEDGER-AUDIT — audit-only ledger entry (bookkeeping, NOT a deflation trial).
# C1 is infrastructure and makes no Sharpe claim, so n_comparisons=0 contributes
# nothing to the cross-PRD deflation N (C1 PRD "Ledger discipline"; A-LEDGER-RUNNERS
# pattern). The verdict is read from the deterministic gate objects above — no
# paraphrase (METHODOLOGY §9).
#
# Idempotent: record_run(skip_if_exists=True) dedups on config_hash, so re-running
# this cell once the entry exists is a no-op (it will NOT double-count N or raise on
# the unique-id invariant). config_hash is the C1-M2 commit SHA — the reconstructed
# convention from data/ledger.yaml's header, because nb13 is a notebook deliverable
# with no runner metadata.json. The committed entry (ledger-2026-06-29-0001) was
# appended out-of-band with this same call so the audit trail exists without
# re-executing this live-data notebook.
from quant.ledger import record_run

_audit = record_run(
    {
        "config_hash": "19a0b405bc8c48bf660c7d890ba5989cf12925ca",
        "started_at": pd.Timestamp.now("UTC").isoformat(),
        "finished_at": pd.Timestamp.now("UTC").isoformat(),
    },
    prd="c1",
    milestone="C1-M2",
    preregistration=".claude/prds/c1-live-data.prd.md#pre-committed-gate",
    n_comparisons=0,
    verdict="gate_passed" if (g1.passed and g2.passed) else "gate_failed",
    agent="human",
    artifacts=["notebooks/13_c1_live_data.ipynb"],
    notes="C1-M2 same-day PIT reader nb13 verdict (G1 no-look-ahead + G2 train/serve "
    "parity). Audit-only; no edge claim (n_comparisons=0).",
)
print("ledger entry:", "already recorded (skipped)" if _audit is None else _audit.id)